# 01. Construction of the Place-Year Analysis Table

The research question for this capstone asks, "To what extent do poverty, ethnic heterogeneity, and residential instability affect the property-crime rate in U.S. cities of 50,000 or more?" That test requires the official crime and census files to occupy a single row per place-year (one city observed in one year). This notebook performs that construction. It follows the numbered extract-and-join sequence used in D205 and the assess–mitigate–reassess sequence used in D206.

Social-disorganization research treats poverty, racial and ethnic heterogeneity, and residential instability as the community conditions through which other compositional measures operate (Ousey, 2000; Stowell, 2007; Kubrin et al., 2022). We therefore assemble those three mediators — the predictors that social-disorganization theory says carry the effect of broader conditions to the outcome —, the FBI property-crime rate, and a short list of controls from files already stored under `data/`. Each official product is read once: DP05 from `place_filter/`, DP03 from `mediator_poverty/`, DP02 and DP04 from `mediator_residential_instability/`, B05002 from `restricted_predictor/`, and the Gazetteer from `controls/`.

The notebook writes three files. `place_year_joined.csv` records the covered universe after the join and the population filter. `place_year_clean.csv` is the analysis table after duplicate-key and missing-predictor drops. `exclusion_log.csv` records every place-year removed from the study universe, with the reason attached.

Several design choices are locked before any value is changed. Table 8 is the published 12-month city table (Federal Bureau of Investigation, n.d.). ACS 5-year place population is the rate denominator (U.S. Census Bureau, n.d.-a). The property-crime count is burglary plus larceny-theft plus motor vehicle theft; arson stays out because reporting is incomplete (Federal Bureau of Investigation, 2019). Township, town, and countywide agency rows that lack a matching ACS place are logged as unmatched.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
if (HERE / "data" / "target").exists():
    ROOT = HERE
elif (HERE.parent / "data" / "target").exists():
    ROOT = HERE.parent
else:
    raise FileNotFoundError("Run this notebook from the Capstone root or from scripts/.")

DATA = ROOT / "data"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = (2022, 2023, 2024)
POP_FLOOR = 50_000
POP_RATIO_LO, POP_RATIO_HI = 0.50, 1.35

TABLE8_PATH = {
    2022: DATA / "target" / "Table_8_Offenses_Known_to_Law_Enforcement_by_State_by_City_2022.csv",
    2023: DATA / "target" / "Table_8_Offenses_Known_to_Law_Enforcement_by_State_by_City_2023.csv",
    2024: DATA / "target" / "CIUS_Table_8_Offenses_Known_to_Law_Enforcement_by_State_by_City_2024.csv",
}

STATE_TO_ABBR = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR",
    "CALIFORNIA": "CA", "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE",
    "DISTRICT OF COLUMBIA": "DC", "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI",
    "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS",
    "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS",
    "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK",
    "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT",
    "VERMONT": "VT", "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI", "WYOMING": "WY",
}
FULL_TO_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
}

# Table 8 normalized city → ACS NAME after the same normalize.
# Countywide PDs (Las Vegas Metro, Honolulu) are intentionally omitted.
ALIASES = {
    ("TN", "nashville"): "nashville davidson metropolitan government balance",
    ("TN", "nashville davidson"): "nashville davidson metropolitan government balance",
    ("TN", "metropolitan nashville police department"): "nashville davidson metropolitan government balance",
    ("KY", "louisville"): "louisville jefferson county metro government balance",
    ("KY", "louisville metro"): "louisville jefferson county metro government balance",
    ("IN", "indianapolis"): "indianapolis balance",
    ("GA", "athens"): "athens clarke county unified government balance",
    ("GA", "athens clarke county"): "athens clarke county unified government balance",
    ("GA", "augusta"): "augusta richmond county consolidated government balance",
    ("GA", "augusta richmond county"): "augusta richmond county consolidated government balance",
    ("KY", "lexington"): "lexington fayette urban county",
    ("GA", "macon"): "macon bibb county",
    ("GA", "macon bibb county"): "macon bibb county",
    ("CA", "ventura"): "san buenaventura ventura",
    ("NC", "charlotte mecklenburg"): "charlotte",
}

SUFFIX = re.compile(r"\b(city|town|village|cdp|borough|municipality|township)\b", re.I)

# Locked Blau groups: Hispanic (any race) + NH race-alone + NH two or more.
# Do not also add Two-or-more sub-splits. IDs differ by ACS year.
BLAU_IDS = {
    2022: {
        "hispanic": "DP05_0073E",
        "nh_white": "DP05_0079E",
        "nh_black": "DP05_0080E",
        "nh_aian": "DP05_0081E",
        "nh_asian": "DP05_0082E",
        "nh_nhpi": "DP05_0083E",
        "nh_other": "DP05_0084E",
        "nh_twoplus": "DP05_0085E",
    },
    2023: {
        "hispanic": "DP05_0076E",
        "nh_white": "DP05_0082E",
        "nh_black": "DP05_0083E",
        "nh_aian": "DP05_0084E",
        "nh_asian": "DP05_0085E",
        "nh_nhpi": "DP05_0086E",
        "nh_other": "DP05_0087E",
        "nh_twoplus": "DP05_0088E",
    },
    2024: {
        "hispanic": "DP05_0090E",
        "nh_white": "DP05_0096E",
        "nh_black": "DP05_0097E",
        "nh_aian": "DP05_0098E",
        "nh_asian": "DP05_0099E",
        "nh_nhpi": "DP05_0100E",
        "nh_other": "DP05_0101E",
        "nh_twoplus": "DP05_0102E",
    },
}

ACS_STABLE = {
    "acs_population": "DP05_0001E",
    "poverty_rate": "DP03_0128PE",
    "unemployment_rate": "DP03_0009PE",
    "median_hh_income": "DP03_0062E",
    "pct_moved_year": "DP02_0081PE",
    "renter_share": "DP04_0047PE",
    "foreign_born_num": "B05002_013E",
    "nativity_den": "B05002_001E",
}

exclusion_rows: list[dict] = []


def log_excl(**kwargs) -> None:
    exclusion_rows.append(kwargs)


def normalize_place(name: object) -> str:
    s = str(name).lower()
    s = re.sub(r"\d+$", "", s)
    s = s.replace(".", "").replace("'", "")
    s = s.replace("saint ", "st ")
    s = SUFFIX.sub("", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def acs_num(s: pd.Series) -> pd.Series:
    out = pd.to_numeric(s, errors="coerce")
    return out.where(out > -1e5)


def flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [" ".join(str(c).split()) for c in df.columns]
    return df


def col_like(df: pd.DataFrame, *needles: str) -> str:
    want = " ".join(needles).lower()
    for c in df.columns:
        if " ".join(str(c).split()).lower() == want:
            return c
    for c in df.columns:
        if all(n.lower() in str(c).lower() for n in needles):
            return c
    raise KeyError(f"No column matching {needles} in {list(df.columns)}")


print("ROOT", ROOT)
print("processed", PROCESSED)


ROOT c:\Users\Owner\OneDrive\Capstone
processed c:\Users\Owner\OneDrive\Capstone\data\processed


## File Gate

Construction begins only when every required official file is already on disk. The gate asserts the presence of Table 8, the ACS place profiles, B05002, and the national place Gazetteer for 2022–2024. If any path is absent, execution stops.

In [2]:
required = list(TABLE8_PATH.values())
for year in YEARS:
    required.extend(
        [
            DATA / "place_filter" / f"ACSDP5Y{year}.DP05.csv",
            DATA / "mediator_poverty" / f"ACSDP5Y{year}.DP03.csv",
            DATA / "mediator_residential_instability" / f"ACSDP5Y{year}.DP02.csv",
            DATA / "mediator_residential_instability" / f"ACSDP5Y{year}.DP04.csv",
            DATA / "restricted_predictor" / f"ACSDT5Y{year}.B05002.csv",
            DATA / "controls" / f"{year}_Gaz_place_national.csv",
        ]
    )
missing = [p for p in required if not p.exists()]
print("required files", len(required))
print("missing", missing)
assert not missing, missing


required files 21
missing []


## D205.1. Offenses Known to Law Enforcement, Table 8

The Federal Bureau of Investigation publishes Table 8 as the city panel of offenses known to law enforcement (Federal Bureau of Investigation, n.d.). Each workbook opens with title rows; we skip those rows (`header=3`) and retain the agency table itself. The state name appears only on the first city of each state block, so we forward-fill `State`. Published footnote digits attached to a state name (`FLORIDA2`) are stripped before the postal-abbreviation map is applied.

We keep city, Table 8 population, the published property-crime total, burglary, larceny-theft, motor vehicle theft, and arson. A `year` column is added, and the three annual files are stacked. Confirmation prints the row count by year, the count of empty city names, and the share of rows that received a state abbreviation.

In [3]:
def load_table8(year: int) -> pd.DataFrame:
    df = flatten_cols(pd.read_csv(TABLE8_PATH[year], header=3))
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]
    df["State"] = df["State"].ffill()
    df = df.dropna(subset=["City"]).copy()
    df["table8_city"] = df["City"].astype(str).str.strip()
    df = df[df["table8_city"].str.len() > 0]
    raw_state = df["State"].astype(str).str.upper().str.strip()
    raw_state = raw_state.str.replace(r"\d+$", "", regex=True).str.strip()
    df["state_abbr"] = raw_state.map(STATE_TO_ABBR)
    df["year"] = year
    df["table8_population"] = acs_num(df[col_like(df, "Population")])
    df["table8_property_crime"] = acs_num(df[col_like(df, "Property", "crime")])
    df["burglary"] = acs_num(df[col_like(df, "Burglary")])
    df["larceny_theft"] = acs_num(df[col_like(df, "Larceny")])
    df["motor_vehicle_theft"] = acs_num(df[col_like(df, "Motor", "vehicle")])
    df["arson"] = acs_num(df[col_like(df, "Arson")])
    keep = [
        "year",
        "state_abbr",
        "table8_city",
        "table8_population",
        "table8_property_crime",
        "burglary",
        "larceny_theft",
        "motor_vehicle_theft",
        "arson",
    ]
    return df[keep]


t8 = pd.concat([load_table8(y) for y in YEARS], ignore_index=True)
print("rows per year")
print(t8.groupby("year").size())
print("empty city names", int((t8["table8_city"].str.len() == 0).sum()))
print("state mapped", int(t8["state_abbr"].notna().sum()), "of", len(t8))
print(t8.head(3))


rows per year
year
2022    7876
2023    8365
2024    8987
dtype: int64
empty city names 0
state mapped 25228 of 25228
   year state_abbr table8_city  table8_population  table8_property_crime  \
0  2022         AL   Abbeville             2390.0                   42.0   
1  2022         AL  Adamsville             4233.0                  230.0   
2  2022         AL     Addison              672.0                   19.0   

   burglary  larceny_theft  motor_vehicle_theft  arson  
0      10.0           30.0                  2.0    0.0  
1      18.0          195.0                 17.0    0.0  
2       8.0            9.0                  2.0    0.0  


Table 8 contributes 7,876 to 8,987 city rows per year. All 25,228 rows received a state abbreviation, and no city name is blank. The three annual files are stacked and ready for the ACS join.

## D205.2. American Community Survey Place Tables

The American Community Survey 5-year Data Profiles supply the place-level social and economic measures that the research question names (U.S. Census Bureau, n.d.-a). DP05 contributes total population and the race and ethnicity counts used later for the Blau index. DP03 contributes poverty, unemployment, and median household income. DP02 contributes the share of residents who lived in a different house one year earlier. DP04 contributes the renter-occupied share. Detailed table B05002 contributes the foreign-born count and its nativity denominator; foreign-born share provides context (Stowell, 2007).

Each annual extract keeps `GEO_ID` and `NAME`. The three years are stacked after the tables for a given year have been joined on `GEO_ID`. Confirmation prints the place count by year—approximately 31,900 to 32,000 places—and the count of places with population of 50,000 or more.

In [4]:
def load_acs_year(year: int) -> pd.DataFrame:
    blau = BLAU_IDS[year]
    dp05_cols = ["GEO_ID", "NAME", ACS_STABLE["acs_population"], *blau.values()]
    dp03_cols = ["GEO_ID", ACS_STABLE["poverty_rate"], ACS_STABLE["unemployment_rate"], ACS_STABLE["median_hh_income"]]
    dp02_cols = ["GEO_ID", ACS_STABLE["pct_moved_year"]]
    dp04_cols = ["GEO_ID", ACS_STABLE["renter_share"]]
    b05_cols = ["GEO_ID", ACS_STABLE["foreign_born_num"], ACS_STABLE["nativity_den"]]

    dp05 = pd.read_csv(DATA / "place_filter" / f"ACSDP5Y{year}.DP05.csv", usecols=dp05_cols, low_memory=False)
    dp03 = pd.read_csv(DATA / "mediator_poverty" / f"ACSDP5Y{year}.DP03.csv", usecols=dp03_cols, low_memory=False)
    dp02 = pd.read_csv(
        DATA / "mediator_residential_instability" / f"ACSDP5Y{year}.DP02.csv",
        usecols=dp02_cols,
        low_memory=False,
    )
    dp04 = pd.read_csv(
        DATA / "mediator_residential_instability" / f"ACSDP5Y{year}.DP04.csv",
        usecols=dp04_cols,
        low_memory=False,
    )
    b05 = pd.read_csv(
        DATA / "restricted_predictor" / f"ACSDT5Y{year}.B05002.csv",
        usecols=b05_cols,
        low_memory=False,
    )

    out = dp05.merge(dp03, on="GEO_ID", how="left")
    out = out.merge(dp02, on="GEO_ID", how="left")
    out = out.merge(dp04, on="GEO_ID", how="left")
    out = out.merge(b05, on="GEO_ID", how="left")
    out["year"] = year
    out["acs_population"] = acs_num(out[ACS_STABLE["acs_population"]])
    out["poverty_rate"] = acs_num(out[ACS_STABLE["poverty_rate"]])
    out["unemployment_rate"] = acs_num(out[ACS_STABLE["unemployment_rate"]])
    out["median_hh_income"] = acs_num(out[ACS_STABLE["median_hh_income"]])
    out["pct_moved_year"] = acs_num(out[ACS_STABLE["pct_moved_year"]])
    out["renter_share"] = acs_num(out[ACS_STABLE["renter_share"]])
    out["foreign_born_share"] = (
        acs_num(out[ACS_STABLE["foreign_born_num"]]) / acs_num(out[ACS_STABLE["nativity_den"]]) * 100
    )
    parts = out["NAME"].str.rsplit(", ", n=1, expand=True)
    out["place_raw"] = parts[0]
    out["state_name"] = parts[1]
    out["state_abbr"] = out["state_name"].map(FULL_TO_ABBR)
    out["is_cdp"] = out["place_raw"].str.contains(r"\bCDP\b", case=False, na=False)
    out["geoid"] = out["GEO_ID"].astype(str).str.replace("1600000US", "", regex=False).str.zfill(7)
    out["match_key"] = out["state_abbr"] + "|" + out["place_raw"].map(normalize_place)
    for label, var in blau.items():
        out[f"blau_{label}"] = acs_num(out[var])
    return out


acs = pd.concat([load_acs_year(y) for y in YEARS], ignore_index=True)
print("ACS places per year")
print(acs.groupby("year").size())
print("ACS pop >= 50,000 per year")
print(acs.loc[acs["acs_population"] >= POP_FLOOR].groupby("year").size())


ACS places per year
year
2022    31894
2023    32033
2024    32038
dtype: int64
ACS pop >= 50,000 per year
year
2022    864
2023    871
2024    876
dtype: int64


Each ACS year contains roughly 32,000 places. Of those, 864 to 876 clear the 50,000-population floor — the universe the research question targets.

## D205.3. Census Place Gazetteer

Land area is required for population density and comes from a separate Census file. The Census Bureau’s national place Gazetteer files supply `GEOID`, land area in square miles, and internal-point latitude and longitude (U.S. Census Bureau, n.d.-b). We load the 2022, 2023, and 2024 files, pad `GEOID` to seven digits so it matches the ACS geography identifier, and confirm that each year contains one row per `GEOID`.

In [5]:
def load_gaz(year: int) -> pd.DataFrame:
    g = pd.read_csv(DATA / "controls" / f"{year}_Gaz_place_national.csv")
    g.columns = [c.strip() for c in g.columns]
    out = pd.DataFrame(
        {
            "year": year,
            "geoid": g["GEOID"].astype(str).str.zfill(7),
            "land_area_sqmi": pd.to_numeric(g["ALAND_SQMI"], errors="coerce"),
            "lat": pd.to_numeric(g["INTPTLAT"], errors="coerce"),
            "lon": pd.to_numeric(g["INTPTLONG"], errors="coerce"),
        }
    )
    return out.drop_duplicates(["year", "geoid"], keep="first")


gaz = pd.concat([load_gaz(y) for y in YEARS], ignore_index=True)
print("Gazetteer rows per year")
print(gaz.groupby("year").size())
print("duplicate GEOID-year", int(gaz.duplicated(["year", "geoid"]).sum()))


Gazetteer rows per year
year
2022    32187
2023    32329
2024    32333
dtype: int64
duplicate GEOID-year 0


The Gazetteer supplies about 32,200 to 32,300 places per year with no duplicate GEOID within a year. Land area is now available for the density calculation after the join.

## D205.4. Match Keys and the Official Join

Table 8 identifies an agency by state and city name. It omits a Census geography identifier. ACS `NAME` values take the form `Ogden city, Utah`. Consequently, the join is a string match. We build one key from the Table 8 state abbreviation plus a normalized city token, and we build the same key from the ACS place token that precedes the comma. Normalization lowercases the string, strips trailing footnote digits, removes the legal suffixes `city`, `town`, `village`, `CDP`, `borough`, `municipality`, and `township`, and collapses punctuation.

A small alias table maps consolidated governments and metro agency titles onto the ACS place name (Nashville-Davidson, Louisville/Jefferson, Indianapolis balance, Charlotte-Mecklenburg, and a short list of peers). Countywide departments that are not ACS places—Las Vegas Metropolitan Police Department and the Honolulu county agency among them—are omitted from that table. Forcing those counts onto a smaller city would assign a multi-jurisdiction numerator to a single-place denominator.

When one key hits both an incorporated place and a census-designated place, we keep the incorporated row. The Gazetteer is then left-joined on `geoid` and `year`. Confirmation reports the match rate and lists unmatched Table 8 cities whose published population is at least 50,000. Those unmatched rows are written to the exclusion log.

In [6]:
def table8_key(row) -> str:
    if pd.isna(row["state_abbr"]):
        return np.nan
    norm = normalize_place(row["table8_city"])
    alias = ALIASES.get((row["state_abbr"], norm), norm)
    return f"{row['state_abbr']}|{alias}"


t8 = t8.copy()
t8["match_key"] = t8.apply(table8_key, axis=1)

for rec in t8.loc[t8["state_abbr"].isna()].itertuples(index=False):
    log_excl(
        year=rec.year,
        state_abbr="",
        place=rec.table8_city,
        geoid="",
        reason="missing_state",
        detail="Table 8 state did not map to a postal abbreviation",
    )

acs_pref = acs.sort_values(
    ["year", "match_key", "is_cdp", "acs_population"],
    ascending=[True, True, True, False],
).drop_duplicates(["year", "match_key"], keep="first")

acs_cols = [
    "year",
    "match_key",
    "GEO_ID",
    "geoid",
    "NAME",
    "place_raw",
    "state_abbr",
    "acs_population",
    "is_cdp",
    "poverty_rate",
    "unemployment_rate",
    "median_hh_income",
    "pct_moved_year",
    "renter_share",
    "foreign_born_share",
    *[f"blau_{k}" for k in next(iter(BLAU_IDS.values()))],
]

joined = t8.merge(acs_pref[acs_cols], on=["year", "match_key"], how="left", suffixes=("", "_acs"))
if "state_abbr_acs" in joined.columns:
    joined["state_abbr"] = joined["state_abbr"].fillna(joined["state_abbr_acs"])
    joined = joined.drop(columns=["state_abbr_acs"])

joined = joined.merge(gaz, on=["year", "geoid"], how="left")

matched = joined["geoid"].notna()
print("Table 8 rows", len(joined))
print("matched to ACS place", int(matched.sum()), f"({matched.mean():.1%})")
print("match rate by year")
print(joined.assign(matched=matched).groupby("year")["matched"].mean())

unmatched = joined.loc[~matched]
print("unmatched Table 8 with table8 pop >= 50,000")
u_big = unmatched.loc[unmatched["table8_population"] >= POP_FLOOR, ["year", "state_abbr", "table8_city", "table8_population"]]
print(len(u_big))
print(u_big.sort_values("table8_population", ascending=False).head(20).to_string(index=False))

for rec in unmatched.itertuples(index=False):
    if pd.notna(rec.table8_population) and rec.table8_population >= POP_FLOOR:
        log_excl(
            year=rec.year,
            state_abbr=rec.state_abbr or "",
            place=rec.table8_city,
            geoid="",
            reason="unmatched_table8",
            detail="No ACS place key match; table8 population >= 50,000",
        )


Table 8 rows 25228
matched to ACS place 22663 (89.8%)
match rate by year
year
2022    0.910234
2023    0.901136
2024    0.885279
Name: matched, dtype: float64
unmatched Table 8 with table8 pop >= 50,000
98
 year state_abbr                              table8_city  table8_population
 2024         NV Las Vegas Metropolitan Police Department          1716565.0
 2023         NV Las Vegas Metropolitan Police Department          1670226.0
 2022         NV Las Vegas Metropolitan Police Department          1667961.0
 2022         HI                                 Honolulu           994799.0
 2024         HI                                 Honolulu           992973.0
 2023         HI                                 Honolulu           987546.0
 2024         NY                             Amherst Town           126184.0
 2023         NY                             Amherst Town           125647.0
 2022         NY                             Amherst Town           123527.0
 2024         NY        

The string-match join reaches about 90 percent of Table 8 rows. The 98 unmatched agencies with population above 50,000 are predominantly township, countywide, or metro departments — Las Vegas Metro, Honolulu, Amherst Town — whose jurisdictions span multiple ACS places. Those rows are logged to preserve the correct denominator.

## D205.5. Study Universe

The research question restricts the sample to U.S. cities with ACS population of 50,000 or more (U.S. Census Bureau, 2021).

We keep a joined row only when ACS 5-year population is at least 50,000. Census-designated places remain only when the Table 8 name matched that place after normalization. We further drop joins whose Table 8 population and ACS population fall outside the ratio interval 0.50–1.35. That guard prevents a countywide agency from being scored with a city denominator. Confirmation prints place-year counts by year, the number of census-designated places retained, and the number of unique places.

In [7]:
pre_filter = joined.copy()
joined_path_n = len(joined)

below = joined["geoid"].notna() & (joined["acs_population"] < POP_FLOOR)
for rec in joined.loc[below].itertuples(index=False):
    if pd.notna(rec.table8_population) and rec.table8_population >= POP_FLOOR:
        log_excl(
            year=rec.year,
            state_abbr=rec.state_abbr or "",
            place=rec.table8_city,
            geoid=rec.geoid,
            reason="acs_pop_below_50000",
            detail=f"acs_population={rec.acs_population}",
        )

ratio = joined["table8_population"] / joined["acs_population"]
ratio_bad = (
    joined["geoid"].notna()
    & (joined["acs_population"] >= POP_FLOOR)
    & ~ratio.between(POP_RATIO_LO, POP_RATIO_HI)
)
for rec in joined.loc[ratio_bad].itertuples(index=False):
    log_excl(
        year=rec.year,
        state_abbr=rec.state_abbr or "",
        place=rec.table8_city,
        geoid=rec.geoid,
        reason="pop_ratio_mismatch",
        detail=f"table8_pop={rec.table8_population}; acs_pop={rec.acs_population}",
    )

universe = joined.loc[
    joined["geoid"].notna()
    & (joined["acs_population"] >= POP_FLOOR)
    & ratio.between(POP_RATIO_LO, POP_RATIO_HI)
].copy()

print("place-years after universe filter", len(universe))
print(universe.groupby("year").size())
print("CDP rows kept (Table 8 matched that place)", int(universe["is_cdp"].sum()))
print("unique geoid-years", universe.groupby(["geoid", "year"]).ngroups)
print("unique places", universe["geoid"].nunique())


place-years after universe filter 2183
year
2022    706
2023    732
2024    745
dtype: int64
CDP rows kept (Table 8 matched that place) 18
unique geoid-years 2182
unique places 771


After the population floor and the population-ratio guard, 2,183 place-years remain across 771 unique places — 706 in 2022, 732 in 2023, and 745 in 2024. Eighteen census-designated places survived because their Table 8 name matched directly. One duplicate GEOID-year pair will be resolved during mitigation.

## D205.6. Offense Coverage

The project plan requires 12 months of property-offense reporting. Table 8 omits a months-reported flag because the table is defined as the 12-month city series (Federal Bureau of Investigation, n.d.). Coverage is therefore the publication rule itself.

We drop any remaining row that has a null value for burglary, larceny-theft, or motor vehicle theft. Zero is a valid count; a null value signals incomplete coverage. Each dropped row is written to the exclusion log. Confirmation prints the number of rows removed and the number of place-years that retain a complete property-offense count.

In [8]:
missing_off = universe[["burglary", "larceny_theft", "motor_vehicle_theft"]].isna().any(axis=1)
for rec in universe.loc[missing_off].itertuples(index=False):
    log_excl(
        year=rec.year,
        state_abbr=rec.state_abbr or "",
        place=rec.table8_city,
        geoid=rec.geoid,
        reason="missing_offense_count",
        detail="missing burglary, larceny-theft, and/or motor vehicle theft",
    )
print("dropped missing offense counts", int(missing_off.sum()))
covered = universe.loc[~missing_off].copy()
print("place-years with complete property-offense counts", len(covered))


dropped missing offense counts 0
place-years with complete property-offense counts 2183


No rows were dropped for missing offense counts. All 2,183 place-years carry complete burglary, larceny-theft, and motor vehicle theft values.

## Locked Race and Ethnicity Groups for the Blau Index

Blau’s macrostructural theory treats heterogeneity as a property of a population’s group mix, not as the size of any one group (Blau, 1977). The index is the probability that two persons drawn at random belong to different groups:

\[
H = 1 - \sum_g p_g^2
\]

where \(p_g\) is group \(g\)’s share of the population. \(H\) equals 0 when one group contains the entire population. It rises as the number of groups increases and as those groups approach equal size. For a partition of \(k\) groups the theoretical maximum is \(1 - 1/k\). With the eight groups locked below, that ceiling is 0.875. Messner (1986) states the same measure—one minus the sum of squared group proportions—when applying Blau’s theorems to societal crime rates. Bruinsma et al. (2013) use this Blau, or Herfindahl, form as the ethnic-heterogeneity term in social-disorganization models and distinguish it from a “percent nonwhite” or “percent minority” share. Unlike a single-group share, the Blau index captures both how many groups are present and how evenly the population is distributed among them.

The index is defined only when the groups \(g\) are mutually exclusive and, taken together, exhaust the population used as the denominator. We therefore lock the groups before any share is computed.

The locked partition is Hispanic of any race, plus the non-Hispanic race-alone groups, plus non-Hispanic two or more races. Detailed two-or-more sub-splits are not added; including them with the two-or-more total would double-count. ACS Data Profile variable identifiers for this block shift by year. The map below is the authority for the compute cell that follows.

| Group | 2022 | 2023 | 2024 |
| --- | --- | --- | --- |
| Hispanic (any race) | `DP05_0073E` | `DP05_0076E` | `DP05_0090E` |
| NH White alone | `DP05_0079E` | `DP05_0082E` | `DP05_0096E` |
| NH Black or African American alone | `DP05_0080E` | `DP05_0083E` | `DP05_0097E` |
| NH American Indian and Alaska Native alone | `DP05_0081E` | `DP05_0084E` | `DP05_0098E` |
| NH Asian alone | `DP05_0082E` | `DP05_0085E` | `DP05_0099E` |
| NH Native Hawaiian and Other Pacific Islander alone | `DP05_0083E` | `DP05_0086E` | `DP05_0100E` |
| NH Some other race alone | `DP05_0084E` | `DP05_0087E` | `DP05_0101E` |
| NH Two or more races | `DP05_0085E` | `DP05_0088E` | `DP05_0102E` |

Each share is \(p_g = n_g / \sum_g n_g\). The denominator is the sum of the eight locked counts, ensuring the denominator stays aligned with the partition.

## D205.7. Derived Measures

The dependent variable is the property-crime rate per 100,000 residents. The count is burglary plus larceny-theft plus motor vehicle theft. The denominator is ACS 5-year place population. Table 8 population serves as a cross-check.

Poverty is the DP03 share of all people below the poverty level. Heterogeneity is the Blau index from the locked groups. Residential instability is the DP02 share of residents who lived in a different house one year earlier. Renter share from DP04 is stored as a companion column. Controls are unemployment, median household income, population, and density (population divided by Gazetteer land area). Foreign-born share is written for later context.

In [9]:
blau_cols = [f"blau_{k}" for k in next(iter(BLAU_IDS.values()))]
group_sum = covered[blau_cols].sum(axis=1)
shares = covered[blau_cols].div(group_sum, axis=0)
covered = covered.copy()
covered["property_crime_count"] = (
    covered["burglary"] + covered["larceny_theft"] + covered["motor_vehicle_theft"]
)
covered["property_crime_rate"] = covered["property_crime_count"] / covered["acs_population"] * 100_000
covered["blau_heterogeneity"] = 1 - (shares**2).sum(axis=1)
covered.loc[group_sum <= 0, "blau_heterogeneity"] = np.nan
covered["density"] = covered["acs_population"] / covered["land_area_sqmi"]
covered["place"] = covered["place_raw"]

print(covered[["property_crime_count", "property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]].describe())
print("Blau outside 0-1", int((~covered["blau_heterogeneity"].between(0, 1) & covered["blau_heterogeneity"].notna()).sum()))


       property_crime_count  property_crime_rate  poverty_rate  \
count           2183.000000          2183.000000   2183.000000   
mean            4507.982593          2319.881577     13.096335   
std            11449.121259          1161.657995      6.326002   
min                0.000000             0.000000      2.600000   
25%             1127.000000          1461.146219      8.000000   
50%             1844.000000          2093.757763     12.200000   
75%             3434.000000          2917.878102     17.100000   
max           196549.000000          9826.695155     37.300000   

       blau_heterogeneity  pct_moved_year  
count         2183.000000     2183.000000  
mean             0.547493       14.567430  
std              0.134863        5.253703  
min              0.057955        3.400000  
25%              0.461271       10.900000  
50%              0.580426       13.900000  
75%              0.651626       17.000000  
max              0.783293       38.100000  
Blau outs

All five derived measures — property-crime rate, poverty rate, Blau heterogeneity, residential mobility, and density — are computed and populated. No Blau value falls outside the 0-to-1 bound. The table is ready for the gap check and the D206 cleaning sequence.

## D205.8. Coverage Check and 200-Row Floor

Table 8 remains the sole source of the target. A supplemental file in `data/cde_gaps/` lists public agency pages for large cities that may be absent from the official table. After the join, we flag ACS places of 50,000 or more that still lack a Table 8 target and that appear in that file. The flags are written to the exclusion log for later review.

The plan requires at least 200 place-years. If that floor is missed, execution stops so a declared rollback (a 25,000 population floor or a burglary-only target) can be applied.

In [10]:
gaps = pd.read_csv(DATA / "cde_gaps" / "bigset_cde_gaps.csv")
gaps["year"] = pd.to_numeric(gaps["year"], errors="coerce")
parts = gaps["city"].astype(str).str.rsplit(", ", n=1, expand=True)
gaps["gap_place"] = parts[0].map(normalize_place)
gaps["gap_state"] = parts[1].astype(str).str.upper().str.strip()
gap_keys = set(zip(gaps["year"], gaps["gap_state"], gaps["gap_place"]))

acs50 = acs.loc[acs["acs_population"] >= POP_FLOOR, ["year", "state_abbr", "place_raw", "geoid", "NAME", "match_key"]].copy()
acs50["norm_place"] = acs50["place_raw"].map(normalize_place)
have_target = set(zip(covered["year"], covered["geoid"]))
gap_candidates = 0
for rec in acs50.itertuples(index=False):
    if (rec.year, rec.geoid) in have_target:
        continue
    if (rec.year, rec.state_abbr, rec.norm_place) in gap_keys or (
        rec.year,
        rec.state_abbr,
        normalize_place(rec.place_raw),
    ) in gap_keys:
        gap_candidates += 1
        log_excl(
            year=rec.year,
            state_abbr=rec.state_abbr or "",
            place=rec.NAME,
            geoid=rec.geoid,
            reason="cde_gap_candidate",
            detail="ACS place >= 50,000 with no official Table 8 target; appears in cde_gaps",
        )

covered["cde_gap_flag"] = 0
print("CDE-gap candidate place-years flagged (not filled)", gap_candidates)
print("n place-years after official join + coverage", len(covered))
if len(covered) < 200:
    raise RuntimeError("n < 200; apply the plan rollback (25,000 floor or burglary-only). Do not add rural places.")
print("n >= 200 gate passed")


CDE-gap candidate place-years flagged (not filled) 10
n place-years after official join + coverage 2183
n >= 200 gate passed


Ten place-years were flagged for later review. The analysis retains 2,183 official Table 8 place-years, well above the 200-row floor.

## Intermediate Write

`place_year_joined.csv` is the covered universe after the official join, the population filter, and the complete-count rule. Duplicate `geoid`–year keys and rows missing a locked predictor have not yet been dropped. Writing the file at this point separates the extract from the D206 cleaning sequence.

In [11]:
joined_keep = [
    "geoid",
    "GEO_ID",
    "place",
    "place_raw",
    "NAME",
    "state_abbr",
    "year",
    "table8_city",
    "table8_population",
    "acs_population",
    "is_cdp",
    "burglary",
    "larceny_theft",
    "motor_vehicle_theft",
    "arson",
    "table8_property_crime",
    "property_crime_count",
    "property_crime_rate",
    "poverty_rate",
    "blau_heterogeneity",
    "pct_moved_year",
    "renter_share",
    "unemployment_rate",
    "median_hh_income",
    "foreign_born_share",
    "land_area_sqmi",
    "density",
    "lat",
    "lon",
    "cde_gap_flag",
]
place_year_joined = covered[joined_keep].copy()
place_year_joined_path = PROCESSED / "place_year_joined.csv"
place_year_joined.to_csv(place_year_joined_path, index=False)
print("wrote", place_year_joined_path, "n=", len(place_year_joined))


wrote c:\Users\Owner\OneDrive\Capstone\data\processed\place_year_joined.csv n= 2183


## D206. Assess

Cleaning begins with a read-only assessment. We inspect column names, grouped summaries, duplicate keys on `geoid` and year and on state, place, and year, and missingness by column. Impossible values are counts below zero, poverty, mobility, and renter shares outside 0–100, a Blau index outside 0–1, and non-positive density.

Outliers on the rate and the three predictors are flagged with the interquartile rule (Tukey, 1977). Outliers are flagged, not removed. The prints in the next cell are the pre-mitigation record.

In [12]:
df = place_year_joined.copy()
df.columns = [re.sub(r"[^0-9a-z_]", "", c.lower().replace(" ", "_")) for c in df.columns]
print("columns")
print(list(df.columns))
print()
print(df.info())
print()
print("key groups")
print(df[["property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]].describe())
print()
print("duplicates geoid+year", int(df.duplicated(["geoid", "year"]).sum()))
print("duplicates state+place+year", int(df.duplicated(["state_abbr", "place", "year"]).sum()))
print()
print("missingness")
print(df.isna().sum())

impossible = pd.DataFrame(
    {
        "neg_count": (df[["burglary", "larceny_theft", "motor_vehicle_theft", "property_crime_count"]] < 0).any(axis=1),
        "poverty_oob": ~df["poverty_rate"].between(0, 100) & df["poverty_rate"].notna(),
        "moved_oob": ~df["pct_moved_year"].between(0, 100) & df["pct_moved_year"].notna(),
        "renter_oob": ~df["renter_share"].between(0, 100) & df["renter_share"].notna(),
        "blau_oob": ~df["blau_heterogeneity"].between(0, 1) & df["blau_heterogeneity"].notna(),
        "density_nonpos": df["density"].notna() & (df["density"] <= 0),
    }
)
print()
print("impossible-value row counts")
print(impossible.sum())


def iqr_flag(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)


print()
print("IQR outlier flags (not dropped)")
for c in ["property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]:
    print(c, int(iqr_flag(df[c]).sum()))


columns
['geoid', 'geo_id', 'place', 'place_raw', 'name', 'state_abbr', 'year', 'table8_city', 'table8_population', 'acs_population', 'is_cdp', 'burglary', 'larceny_theft', 'motor_vehicle_theft', 'arson', 'table8_property_crime', 'property_crime_count', 'property_crime_rate', 'poverty_rate', 'blau_heterogeneity', 'pct_moved_year', 'renter_share', 'unemployment_rate', 'median_hh_income', 'foreign_born_share', 'land_area_sqmi', 'density', 'lat', 'lon', 'cde_gap_flag']

<class 'pandas.core.frame.DataFrame'>
Index: 2183 entries, 17 to 25202
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   geoid                  2183 non-null   object 
 1   geo_id                 2183 non-null   object 
 2   place                  2183 non-null   object 
 3   place_raw              2183 non-null   object 
 4   name                   2183 non-null   object 
 5   state_abbr             2183 non-null   object 
 6   yea

The pre-mitigation table has 30 columns and 2,183 rows. One duplicate GEOID-year key exists. Arson is the only column with missing values (87 nulls); it stays out of the property-crime count. The three locked predictors and the rate have no missing values, no impossible values, and no non-positive densities. Interquartile flags mark 58 place-years on the rate; those flags are retained.

## D206. Mitigate

Mitigation is limited to the defects the assessment named. When two Table 8 agencies share one `geoid` and year, we keep the row with the larger Table 8 population and log the other as a duplicate. We then drop any row that is missing the property-crime rate or any of the three locked predictors. Year is stored as an integer; rates and income are stored as floating-point numbers.

The three mediators stay in their original units.

In [13]:
work = place_year_joined.sort_values(
    ["geoid", "year", "table8_population"], ascending=[True, True, False]
).copy()
dup_mask = work.duplicated(["geoid", "year"], keep="first")
for rec in work.loc[dup_mask].itertuples(index=False):
    log_excl(
        year=rec.year,
        state_abbr=rec.state_abbr or "",
        place=rec.place,
        geoid=rec.geoid,
        reason="duplicate_geoid_year",
        detail=f"dropped extra Table 8 agency {rec.table8_city}",
    )
work = work.loc[~dup_mask].copy()

need = ["property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]
miss_pred = work[need].isna().any(axis=1)
for rec in work.loc[miss_pred].itertuples(index=False):
    missing_cols = [c for c in need if pd.isna(getattr(rec, c))]
    log_excl(
        year=rec.year,
        state_abbr=rec.state_abbr or "",
        place=rec.place,
        geoid=rec.geoid,
        reason="missing_target_or_predictor",
        detail=",".join(missing_cols),
    )
work = work.loc[~miss_pred].copy()
work["year"] = work["year"].astype(int)
for c in [
    "property_crime_rate",
    "poverty_rate",
    "blau_heterogeneity",
    "pct_moved_year",
    "renter_share",
    "unemployment_rate",
    "median_hh_income",
    "foreign_born_share",
    "density",
    "acs_population",
]:
    work[c] = pd.to_numeric(work[c], errors="coerce")

print("after mitigate", len(work), "place-years")
print(work.groupby("year").size())


after mitigate 2182 place-years
year
2022    706
2023    731
2024    745
dtype: int64


One duplicate key was removed, bringing the table from 2,183 to 2,182 place-years. No rows were lost to missing predictors. The year split is 706 (2022), 731 (2023), 745 (2024).

## D206. Reassess and Save

The assess block is repeated on the mitigated frame. We confirm that `geoid` and year are unique, that the locked columns contain no missing values, and that the place-year count remains at or above 200. The analysis table is written to `data/processed/place_year_clean.csv`. The exclusion log is written to `data/processed/exclusion_log.csv`. Those two files, together with the earlier joined table, close the D205–D206 sequence.

In [14]:
clean = work.copy()
print(clean.info())
print()
print(clean[["property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]].describe())
print()
print("duplicates geoid+year", int(clean.duplicated(["geoid", "year"]).sum()))
print("missing on H0 columns")
print(clean[["property_crime_rate", "poverty_rate", "blau_heterogeneity", "pct_moved_year"]].isna().sum())
print("n place-years", len(clean), "n places", clean["geoid"].nunique())

excl = pd.DataFrame(exclusion_rows)
if len(excl):
    excl = excl[["year", "state_abbr", "place", "geoid", "reason", "detail"]]
excl_path = PROCESSED / "exclusion_log.csv"
clean_path = PROCESSED / "place_year_clean.csv"
clean.to_csv(clean_path, index=False)
excl.to_csv(excl_path, index=False)
print("wrote", clean_path, "n=", len(clean))
print("wrote", excl_path, "n=", len(excl))
print(excl["reason"].value_counts() if len(excl) else "no exclusions")
if len(clean) < 200:
    raise RuntimeError("clean n < 200; record a rollback before continuing.")
print("01 done")


<class 'pandas.core.frame.DataFrame'>
Index: 2182 entries, 17 to 25202
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   geoid                  2182 non-null   object 
 1   GEO_ID                 2182 non-null   object 
 2   place                  2182 non-null   object 
 3   place_raw              2182 non-null   object 
 4   NAME                   2182 non-null   object 
 5   state_abbr             2182 non-null   object 
 6   year                   2182 non-null   int64  
 7   table8_city            2182 non-null   object 
 8   table8_population      2182 non-null   float64
 9   acs_population         2182 non-null   float64
 10  is_cdp                 2182 non-null   object 
 11  burglary               2182 non-null   float64
 12  larceny_theft          2182 non-null   float64
 13  motor_vehicle_theft    2182 non-null   float64
 14  arson                  2095 non-null   float64
 15  table8_

The final table has 2,182 place-years across 771 places. All 30 columns are present, GEOID-year keys are unique, and the four locked columns carry no missing values. Arson retains 2,095 non-null values as a companion column. The analysis table and exclusion log are written.

## References

Baranauskas, A. J. (2020). Exploring the social construction of crime by neighborhood. *Sociological Focus*. https://doi.org/10.1080/00380237.2020.1730280

Blau, P. M. (1977). *Inequality and heterogeneity: A primitive theory of social structure*. Free Press.

Bruinsma, G. J. N., Pauwels, L. J. R., Weerman, F. M., & Bernasco, W. (2013). Social disorganization, social capital, collective efficacy and the spatial distribution of crime and offenders: An empirical test of six neighbourhood models for a Dutch city. *British Journal of Criminology, 53*(5), 942–963. https://doi.org/10.1093/bjc/azt030

Federal Bureau of Investigation. (2019). *Property crime*. Uniform Crime Reporting Program. https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/property-crime

Federal Bureau of Investigation. (n.d.). *Crime Data Explorer*. https://cde.ucr.cjis.gov/

Kubrin, C. E., Branic, N., & Hipp, J. R. (2022). (Re)conceptualizing neighborhood ecology in social disorganization theory: From a variable-centered approach to a neighborhood-centered approach. *Crime & Delinquency*. https://doi.org/10.1177/00111287211041527

Lundman, R. J. (2003). The newsworthiness and selection bias in news about murder: Comparative and relative effects of novelty and race and gender typifications on newspaper coverage of homicide. *Sociological Forum, 18*(3), 357–386. https://doi.org/10.1023/A:1025713518156

Messner, S. F. (1986). Modernization, structural characteristics, and societal rates of crime: An application of Blau's macrosociological theory. *The Sociological Quarterly, 27*(1), 27–41. https://doi.org/10.1111/j.1533-8525.1986.tb00247.x

Ousey, G. C. (2000). Explaining regional and urban variation in crime: A review of research. In *Criminal Justice 2000* (Vol. 1). National Institute of Justice. https://www.ojp.gov/ncjrs/virtual-library/abstracts/explaining-regional-and-urban-variation-crime-review-research

Stowell, J. I. (2007). *Immigration and crime: The effects of immigration on criminal behavior*. LFB Scholarly Publishing.

Tukey, J. W. (1977). *Exploratory data analysis*. Addison-Wesley.

U.S. Census Bureau. (2021). *2010 Census urban and rural classification and urban area criteria*. https://www.census.gov/programs-surveys/geography/guidance/geo-areas/urban-rural/2010-urban-rural.html

U.S. Census Bureau. (2022, December 22). *Redefining urban areas following the 2020 Census*. https://www.census.gov/newsroom/blogs/random-samplings/2022/12/redefining-urban-areas-following-2020-census.html

U.S. Census Bureau. (n.d.-a). *American Community Survey (ACS)*. https://www.census.gov/programs-surveys/acs

U.S. Census Bureau. (n.d.-b). *Gazetteer files*. https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html
